DAY 6 GOALS

    Build 1D CNN

    Build LSTM

    Build GRU

    Evaluate all 3 models

    Plot ROC curves + AUC

    Compare all deep learning models

### IMPORTANT (RESHAPING)

Advanced neural models need special input format:
    ML + DNN inputs:

    (samples, features)

    CNN + LSTM + GRU inputs:

    (samples, timesteps, features_per_step)

In [ ]:
X_train_dl = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_dl  = X_test.reshape((X_test.shape[0],  X_test.shape[1],  1))


In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

import tensorflow as tf
from tensorflow.keras import layers, models


In [ ]:
# Prepare Deep Learning Input
# Reshape for Conv1D / LSTM / GRU
X_train_dl = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_dl = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

input_shape = (X_train_dl.shape[1], 1)


In [ ]:
# MODEL 1: 1D CNN for DDoS Detection
cnn = models.Sequential([
    layers.Conv1D(64, kernel_size=3, activation='relu', input_shape=input_shape),
    layers.Conv1D(64, kernel_size=3, activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.3),

    layers.Conv1D(128, kernel_size=3, activation='relu'),
    layers.GlobalMaxPooling1D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(1, activation='sigmoid')
])

cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.summary()

history_cnn = cnn.fit(
    X_train_dl, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=1024,
    verbose=1
)


MODEL 2: LSTM (Sequence Model)

In [ ]:
lstm = models.Sequential([
    layers.LSTM(128, return_sequences=True, input_shape=input_shape),
    layers.Dropout(0.3),

    layers.LSTM(64),
    layers.Dropout(0.3),

    layers.Dense(1, activation='sigmoid')
])

lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm.summary()

history_lstm = lstm.fit(
    X_train_dl, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=512,
    verbose=1
)


MODEL 3: GRU (Faster than LSTM)

In [ ]:
gru = models.Sequential([
    layers.GRU(128, return_sequences=True, input_shape=input_shape),
    layers.Dropout(0.3),

    layers.GRU(64),
    layers.Dropout(0.3),

    layers.Dense(1, activation='sigmoid')
])

gru.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
gru.summary()

history_gru = gru.fit(
    X_train_dl, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=512,
    verbose=1
)


Evaluation Helper Function

In [ ]:
def evaluate_dl_model(model, X_test, y_test, name="Model"):
    preds = (model.predict(X_test) > 0.5).astype("int32")

    print(f"\n Results for {name}")
    print(classification_report(y_test, preds))

    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{name} Confusion Matrix")
    plt.show()


Evaluate All Models

In [ ]:
evaluate_dl_model(cnn,  X_test_dl, y_test, "1D CNN")
evaluate_dl_model(lstm, X_test_dl, y_test, "LSTM")
evaluate_dl_model(gru,  X_test_dl, y_test, "GRU")


ROC CURVE + AUC SCORE

In [ ]:
plt.figure(figsize=(8,6))

for model, label in [(cnn, "CNN"), (lstm, "LSTM"), (gru, "GRU")]:
    y_score = model.predict(X_test_dl).ravel()
    fpr, tpr, _ = roc_curve(y_test, y_score)
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{label} (AUC={auc_score:.3f})")

plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()


Save the Advanced Models

In [ ]:
cnn.save("cnn_ddos_model.h5")
lstm.save("lstm_ddos_model.h5")
gru.save("gru_ddos_model.h5")

print("Models saved!")
